### 0. Dependancies

In [3]:
!pip install protobuf
!pip install sentencepiece

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\kysel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\kysel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### 1. Loading the dataset

In [1]:
import pandas as pd
import os
import json
from pathlib import Path

# Path to SMOL dataset
smol_path = Path('smol')

# Subfolders in SMOL dataset
subfolders = ['gatitos', 'smoldoc', 'smolsent']

# Function to read and combine all SMOL files
def load_smol_data():
    data = []
    for subfolder in subfolders:
        subfolder_path = smol_path / subfolder
        for file in subfolder_path.glob('*.jsonl'):
            with open(file, 'r', encoding='utf-8') as f:
                for line in f:
                    json_obj = json.loads(line.strip())
                    # choose correct source key per subfolder
                    if subfolder in ('gatitos', 'smolsent'):
                        source = json_obj.get('src', '')
                    else:  # smoldoc
                        source = json_obj.get('srcs', '')

                    # allow for either 'trgs' or 'trg' for target as a fallback
                    target = json_obj.get('trgs', json_obj.get('trg', ''))

                    data.append({
                        'source': source,
                        'target': target,
                        'category': subfolder
                    })
    
    return pd.DataFrame(data)

# Load the data
smol_df = load_smol_data()

# Format for mBART-50
formatted_data = {
    'translation': []
}

for _, row in smol_df.iterrows():
    formatted_data['translation'].append({
        'source': row['source'],
        'target': row['target']
    })

# Convert to DataFrame for easier handling
formatted_df = pd.DataFrame(formatted_data)

# Display first few entries and statistics
print("Total number of samples:", len(formatted_df))
print("\nSamples per category:")
print(smol_df['category'].value_counts())
formatted_df.head()

Total number of samples: 1498527

Samples per category:
category
gatitos     1299637
smolsent     144958
smoldoc       53932
Name: count, dtype: int64


,translation
0,"{'source': 'Qafar', 'target': ['Afar']}"
1,"{'source': 'Anu', 'target': ['I']}"
2,"{'source': 'Yeey', 'target': ['Yes']}"
3,"{'source': 'Meqeh', 'target': ['Yes']}"
4,"{'source': 'Salaam qalaykum', 'target': ['Hell..."


### 2. Load mBART-50 model

In [2]:
from transformers import MBartForConditionalGeneration, MBart50Tokenizer

# Load mBART-50 tokenizer and model
tokenizer = MBart50Tokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

C:\Users\kysel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
